# Phase 2.5 — Empirical timing calibration (n=3 stabilized greedy)

**Purpose:** measure the n=3 solver's **throughput (nodes/sec)** *and* **memory (MB/worker)** so we
can size the full sweep and pick budget tiers + how many parallel workers fit in RAM. Timing run,
not a correctness run.

**How to run on Colab**
1. Runtime → Change runtime type → **CPU** (High-RAM if offered).
2. Edit **only the CONFIG cell**. Set `REPO_URL`/`BRANCH`; leave `PROBE_WORKERS=None` to auto-fill the
   box and `RUN_TAG=None` for an auto trace tag (or pin either), pick `PROBE_BUDGET`, sample sizes.
3. Runtime → **Run all**. Outputs are **uniquely named per run** — `results/calibration_<TAG>.jsonl`
   (metrics) and `results/paths_<TAG>.jsonl` (each *solved* probe's retraced move+state path), where
   `<TAG>` encodes arm/budget/workers/time. Both are copied to Drive immediately (a stopped run still
   persists). Resume reads **every** prior result file (local + Drive) so finished probes are never
   recomputed; only new work runs.
4. Read the **SUMMARY** table (it aggregates across all runs), run the **VERIFY** cell (independently
   re-checks every solved path and prints the exact substitution moves), download the `*_<TAG>.jsonl`
   files (or grab the Drive copies), and share them.

**Picking up new code after a push** (no runtime recreation): with `UPDATE_REPO=True` the setup cell
`git pull`s the branch every run, so just **Runtime → Restart runtime → Run all**. A *restart* gives a
fresh kernel (re-imports the updated `greedy_nrel`/`calibrate_probe`) while keeping the disk (no
re-clone, no re-`pip`), and the pull updates the files. You only need *Disconnect and delete runtime*
if the environment itself breaks. The setup cell prints the current commit hash — check it matches.

> Prereq: the branch must contain `greedy_nrel.py`, `stabilize.py`, `calibrate_probe.py`, and this
> notebook (committed & pushed).
> Memory note: each worker's `visited` grows with the budget (**~14 KB/node measured for n=3**), so
> **RAM, not cores, caps parallelism** at high budgets — the summary computes the ceiling and the
> RAM-balanced budget tier for you.


In [2]:
# ==================== PHASE 2.5 CALIBRATION — CONFIG (edit ONLY this cell) ====================
# Goal: measure n=3 throughput (nodes/sec) AND memory (MB/worker) to size the full sweep.

# --- repo / environment (Colab) ---
REPO_URL          = "https://github.com/Avi161/ACSolverX.git"
BRANCH            = "test/stable-ac-moves"
REPO_DIR          = "ACSolverX"
CLONE             = True               # clone the repo if REPO_DIR is absent
UPDATE_REPO       = True               # if REPO_DIR already exists, git-pull latest (reset --hard to
                                       #   origin/BRANCH) so a plain RESTART picks up new pushes WITHOUT
                                       #   deleting/recreating the runtime. Untracked files (your
                                       #   results/*.jsonl) are preserved. Set False to pin the current code.
MOUNT_DRIVE       = True
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/acsolverx_calibration"

# --- run identity (traceability): output files are named results/calibration_<TAG>.jsonl + paths_<TAG>.jsonl
RUN_TAG           = None              # None = AUTO tag "<arms>_b<budget>k_w<workers>_<YYYYmmdd-HHMMSS>",
                                       #   unique per run so nothing overwrites and each file traces back
                                       #   to its arm/budget/worker-count/time. Every record also carries
                                       #   run_tag + n_workers. Pin a string (e.g. "sweep1") to reuse one
                                       #   file across runs. Resume reads ALL prior files regardless of tag,
                                       #   so a unique tag still skips already-finished probes.

# --- what to probe ---
N_GEN                 = 3
ARMS                  = ["r1"]        # z=w arm(s). "r1" = representative; add "r2" for the heaviest arm.
MAX_LEN               = 24
USE_NULL_REVERT_BLOCK = True

# --- parallelism (fixes "box RAM barely used"): run probes across cores ---
PROBE_WORKERS = None                   # None = AUTO: fill every core that fits in RAM at PROBE_BUDGET
                                       #   = min(cores, (box_RAM - RAM_SAFETY_GB) / (~14KB/node*PROBE_BUDGET)).
                                       #   On an 8-core/51GB box @ 200k this resolves to 8 (all cores).
                                       #   Pin an int to override (8 = all cores; 1 = single-core baseline).
                                       #   NOTE: with 8 cores, ~8 workers is the CPU ceiling; to actually
                                       #   fill the RAM budget you must raise PROBE_BUDGET (see SUMMARY's
                                       #   RAM-balanced tier), not add workers. ~14 KB/node measured (n=3).
RAM_SAFETY_GB = 10                     # leave this many GB free as a crash-safety margin. Auto-worker
                                       #   sizing and the projection only ever budget (box_RAM - this),
                                       #   e.g. 51GB box -> plan against ~41GB, never the full 51.

# --- budgets ---
PROBE_BUDGET      = 500_000            # nodes each HARD probe runs to. Throughput is ~stable with depth
                                       #   now, so 200k gives a solid nodes/sec fast. Raise toward your
                                       #   target tier to also measure memory at that tier.
CANDIDATE_BUDGETS = [100_000, 1_000_000]

# --- samples ---
HARD_DATASET = "ms_reps_unsolved"      # 261 hard reps -> data/ms_unsolved_reps/ms_reps_unsolved.txt
N_HARD       = 261
HARD_IDX     = None                    # None -> evenly spread; or explicit list e.g. [0, 65, 130, 195, 260]

EASY_DATASET = "1190MS"
N_EASY       = 3
EASY_IDX     = None
EASY_BUDGET  = 100_000

# --- projection: (label, total #presentations, approx #budget-exhausting "unsolved" cases) ---
PROJECT_OVER = [
    ("MS(1190)",          1190, 550),
    ("unsolved reps 261",  261, 261),
]
N_ARMS_TOTAL       = 4                 # x,y,r1,r2 -> whole-experiment compute multiplier
BOX_RAM_GB_OVERRIDE = None             # None -> auto-detect the box RAM; set a number to override
# =============================================================================================
print("config loaded.")


config loaded.


In [3]:
# ==================== SETUP (clone / install / mount / import / detect box / name this run) ====================
import os, sys, subprocess, ast, time, json, glob, resource, platform, statistics, multiprocessing as mp
def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        # pull latest without recloning; reset --hard leaves untracked results/*.jsonl intact
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")   # <-- confirm which commit you're running
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while REPO_ROOT != "/" and not os.path.isdir(os.path.join(REPO_ROOT, "envs")):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

ONEGEN = os.path.join(REPO_ROOT, "experiments", "stable_ac", "one_generator")
sys.path.insert(0, ONEGEN)
import greedy_nrel as gn
import stabilize as stab

RESULTS_DIR = os.path.join(ONEGEN, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

RAW = {
    "1190MS":           os.path.join(REPO_ROOT, "data", "1190MS.txt"),
    "ms_reps_unsolved": os.path.join(REPO_ROOT, "data", "ms_unsolved_reps", "ms_reps_unsolved.txt"),
}
def load_raw(name):
    with open(RAW[name]) as f:
        return [ast.literal_eval(l) for l in f if l.strip()]

# --- history helpers: resume/summary/verify read EVERY prior result file (local + Drive), any tag ---
def result_files(prefix):
    dirs = [RESULTS_DIR] + ([DRIVE_RESULTS_DIR] if (IN_COLAB and MOUNT_DRIVE) else [])
    return sorted(p for d in dirs for p in glob.glob(os.path.join(d, prefix + "*.jsonl")))
def read_jsonl(paths):
    out = []
    for p in paths:
        try:
            for l in open(p):
                l = l.strip()
                if l:
                    try: out.append(json.loads(l))
                    except Exception: pass
        except Exception: pass
    return out

# --- box + auto worker count (needed for the run tag) ---
CORES = os.cpu_count() or 1
def _box_ram_gb():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return int(line.split()[1]) / (1024.0 * 1024.0)  # KB -> GB
    except Exception:
        return None
BOX_RAM_GB = BOX_RAM_GB_OVERRIDE or _box_ram_gb()
usable_gb = max(1.0, (BOX_RAM_GB or 12.0) - RAM_SAFETY_GB)          # keep RAM_SAFETY_GB free (crash margin)
if PROBE_WORKERS:
    n_workers = int(PROBE_WORKERS)
else:
    _gb_per   = PROBE_BUDGET * 14.0 / (1024.0 * 1024.0)            # ~14 KB/node measured (n=3) -> GB/worker
    n_workers = max(1, min(CORES, int(usable_gb / _gb_per) if _gb_per > 0 else CORES))

# --- this run's identity -> unique, traceable output filenames ---
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")                         # box clock (UTC on Colab)
RUN_TAG_RESOLVED = RUN_TAG or f"{'-'.join(ARMS)}_b{PROBE_BUDGET//1000}k_w{n_workers}_{RUN_STAMP}"
CALIB_PATH = os.path.join(RESULTS_DIR, f"calibration_{RUN_TAG_RESOLVED}.jsonl")
PATHS_PATH = os.path.join(RESULTS_DIR, f"paths_{RUN_TAG_RESOLVED}.jsonl")

print(f"repo : {REPO_ROOT}")
print(f"tag  : {RUN_TAG_RESOLVED}")
print(f"out  : {os.path.basename(CALIB_PATH)}  +  {os.path.basename(PATHS_PATH)}   (in {RESULTS_DIR})")
print(f"box  : {CORES} cores, RAM {BOX_RAM_GB and round(BOX_RAM_GB,1)}GB -> budget {usable_gb:.0f}GB "
      f"({RAM_SAFETY_GB}GB safety) ; workers={n_workers} "
      f"(~{n_workers*PROBE_BUDGET*14.0/1024/1024:.0f}GB peak @ {PROBE_BUDGET:,} nodes)")


Mounted at /content/drive
repo : /content/ACSolverX
tag  : r1_b500k_w6_20260703-051042
out  : calibration_r1_b500k_w6_20260703-051042.jsonl  +  paths_r1_b500k_w6_20260703-051042.jsonl   (in /content/ACSolverX/experiments/stable_ac/one_generator/results)
box  : 8 cores, RAM 51.0GB -> budget 41GB (10GB safety) ; workers=6 (~40GB peak @ 500,000 nodes)


In [4]:
# ==================== WARM UP numba (parent compiles; forked workers inherit the warm code) ====================
_warm = stab.stabilize_flat(load_raw("1190MS")[0], "r1")
t0 = time.time()
gn.solve_one(_warm, n_gen=N_GEN, max_len=MAX_LEN, max_nodes=2000)
print(f"warmup (numba JIT of all primitives) done in {time.time()-t0:.1f}s")


warmup (numba JIT of all primitives) done in 2.9s


In [5]:
# ==================== BUILD PROBE INDICES ====================
def spread_idx(n_total, k):
    if k <= 1: return [0]
    if k >= n_total: return list(range(n_total))
    return sorted({round(i * (n_total - 1) / (k - 1)) for i in range(k)})

hard_raw = load_raw(HARD_DATASET)
easy_raw = load_raw(EASY_DATASET)
hard_idx = HARD_IDX if HARD_IDX is not None else spread_idx(len(hard_raw), N_HARD)
easy_idx = EASY_IDX if EASY_IDX is not None else spread_idx(640, N_EASY)  # 1190MS solved region is idx<640
print(f"HARD ({HARD_DATASET}, {len(hard_raw)} lines) idx:", hard_idx)
print(f"EASY ({EASY_DATASET}) idx:", easy_idx)


HARD (ms_reps_unsolved, 261 lines) idx: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213,

In [6]:
# ==================== RUN PROBES (parallel, resumable across ALL prior files, incremental Drive copy) ====================
from calibrate_probe import probe as _probe_worker   # importable -> picklable under fork AND spawn

def _persist(rec):
    rec["run_tag"] = RUN_TAG_RESOLVED; rec["n_workers"] = n_workers   # stamp every line -> traces to its run
    prec = rec.pop("path_record", None)               # solved probes carry the retraced move+state path
    if prec is not None:                              # write the PATH first: a crash before the metrics line
        prec["run_tag"] = RUN_TAG_RESOLVED            #   then just re-runs the probe (no orphaned metric row)
        with open(PATHS_PATH, "a") as f:
            f.write(json.dumps(prec) + "\n"); f.flush(); os.fsync(f.fileno())
    with open(CALIB_PATH, "a") as f:
        f.write(json.dumps(rec) + "\n"); f.flush(); os.fsync(f.fileno())
    if IN_COLAB and MOUNT_DRIVE:
        import shutil
        shutil.copy(CALIB_PATH, os.path.join(DRIVE_RESULTS_DIR, os.path.basename(CALIB_PATH)))
        if os.path.exists(PATHS_PATH):
            shutil.copy(PATHS_PATH, os.path.join(DRIVE_RESULTS_DIR, os.path.basename(PATHS_PATH)))
    print(f"  [{rec['kind']:4}] {rec['dataset']:16} idx {rec['idx']:4} z={rec['arm']:2}: "
          f"solved={rec['solved']!s:5} nodes={rec['nodes_explored']:8} {rec['wall_time_s']:7.1f}s "
          f"{rec['nodes_per_sec']:6}/s  rss={rec['peak_rss_mb']:.0f}MB  reverts={rec['revert_hits']}")

# resume across ALL prior result files (local + Drive, any tag): a probe is "done" (skip) only if it was
# UNSOLVED, or it solved AND its path is persisted -- so an already-solved probe missing a path re-runs
# (cheap) to emit it, while budget-exhausting hard probes computed in any prior run stay skipped.
have_path = {(r["dataset"], r["idx"], r["arm"]) for r in read_jsonl(result_files("paths"))
             if all(k in r for k in ("dataset", "idx", "arm"))}
done = set()
for r in read_jsonl(result_files("calibration")):
    try:
        if (not r["solved"]) or ((r["dataset"], r["idx"], r["arm"]) in have_path):
            done.add((r["kind"], r["dataset"], r["idx"], r["arm"], r["budget_nodes"]))
    except Exception:
        pass

tasks = []
for arm in ARMS:
    for idx in hard_idx:
        if ("hard", HARD_DATASET, idx, arm, PROBE_BUDGET) not in done:
            tasks.append(("hard", HARD_DATASET, idx, arm, PROBE_BUDGET, hard_raw[idx], N_GEN, MAX_LEN, USE_NULL_REVERT_BLOCK))
    for idx in easy_idx:
        if ("easy", EASY_DATASET, idx, arm, EASY_BUDGET) not in done:
            tasks.append(("easy", EASY_DATASET, idx, arm, EASY_BUDGET, easy_raw[idx], N_GEN, MAX_LEN, USE_NULL_REVERT_BLOCK))
print(f"{len(tasks)} probes to run on {n_workers} worker(s)  ({len(done)} already done across prior files)\n")

t0 = time.time()
if tasks and n_workers > 1:
    ctx = mp.get_context("fork")           # Colab/Linux: fresh child per task (clean per-probe RSS), warm numba
    with ctx.Pool(n_workers, maxtasksperchild=1) as pool:
        for rec in pool.imap_unordered(_probe_worker, tasks):
            _persist(rec)
else:
    for t in tasks:
        _persist(_probe_worker(t))
print(f"\ndone in {time.time()-t0:.0f}s wall.  metrics -> {CALIB_PATH}  |  paths -> {PATHS_PATH}"
      + (f"  (+Drive: {DRIVE_RESULTS_DIR}/)" if (IN_COLAB and MOUNT_DRIVE) else ""))


194 probes to run on 6 worker(s)  (350 already done across prior files)

  [hard] ms_reps_unsolved idx   69 z=r1: solved=False nodes=  500000   836.3s    598/s  rss=2726MB  reverts=1946
  [hard] ms_reps_unsolved idx   72 z=r1: solved=False nodes=  500000   838.6s    596/s  rss=2728MB  reverts=2011
  [hard] ms_reps_unsolved idx   68 z=r1: solved=False nodes=  500000   861.0s    581/s  rss=2841MB  reverts=1969
  [hard] ms_reps_unsolved idx   70 z=r1: solved=False nodes=  500000  1005.5s    497/s  rss=3683MB  reverts=1130
  [hard] ms_reps_unsolved idx   71 z=r1: solved=False nodes=  500000  1058.1s    473/s  rss=3557MB  reverts=1440
  [hard] ms_reps_unsolved idx   66 z=r1: solved=False nodes=  500000  1124.8s    445/s  rss=5592MB  reverts=1015
  [hard] ms_reps_unsolved idx   74 z=r1: solved=False nodes=  500000   783.7s    638/s  rss=2740MB  reverts=2746
  [hard] ms_reps_unsolved idx   75 z=r1: solved=False nodes=  500000   780.3s    641/s  rss=1921MB  reverts=3024
  [hard] ms_reps_unsolv

In [7]:
# ==================== SUMMARY & MEMORY-AWARE SWEEP PROJECTION (aggregates ALL runs) ====================
# dedup across every prior calibration_*.jsonl (local + Drive) by (kind,dataset,idx,arm,budget); last wins.
_by = {}
for r in read_jsonl(result_files("calibration")):
    try: _by[(r["kind"], r["dataset"], r["idx"], r["arm"], r["budget_nodes"])] = r
    except Exception: pass
recs = list(_by.values())
hard = [r for r in recs if r["kind"] == "hard"]
easy = [r for r in recs if r["kind"] == "easy"]
print(f"probes: {len(hard)} hard, {len(easy)} easy   (arms: {sorted(set(r['arm'] for r in recs))}, "
      f"runs: {sorted(set(r.get('run_tag','?') for r in recs))})\n")

exh     = [r["nodes_per_sec"] for r in hard if r["exhausted_budget"]]
allhard = [r["nodes_per_sec"] for r in hard]
nps_est = statistics.median(exh) if exh else (statistics.median(allhard) if allhard else 0)
kbpn    = statistics.median([r["peak_rss_mb"] * 1024.0 / r["nodes_explored"]
                             for r in hard if r["nodes_explored"] > 0]) if hard else 0
peak_mb = max((r["peak_rss_mb"] for r in hard), default=0)
ram_gb  = BOX_RAM_GB_OVERRIDE or BOX_RAM_GB or 12.0
budget_ram = max(1.0, ram_gb - RAM_SAFETY_GB)      # plan against this, not the full box (crash margin)

print("THROUGHPUT (nodes/sec, per core):")
if allhard:
    print(f"  hard all      : median {statistics.median(allhard):.0f}  min {min(allhard):.0f}  max {max(allhard):.0f}")
print(f"  hard exhausting: median {nps_est:.0f}   <-- used for the projection ({len(exh)}/{len(hard)} exhausted budget)")
if not exh:
    print("  WARNING: no hard probe ran to full budget -> nodes/sec is a SOLVE-time number (optimistic).")
    print("           raise PROBE_BUDGET or pick harder HARD_IDX so probes exhaust the budget.")
if easy:
    print(f"  easy solved   : median wall {statistics.median([r['wall_time_s'] for r in easy]):.2f}s (small addend)")
print(f"\nMEMORY (per worker): peak at {PROBE_BUDGET:,} nodes = {peak_mb:.0f} MB ; rate ~{kbpn:.1f} KB/node")
print(f"  box RAM {ram_gb:.0f} GB ; planning against {budget_ram:.0f} GB ({RAM_SAFETY_GB} GB safety margin left free)")

# The budget tier that keeps ALL cores busy without spilling the RAM budget: budget_ram/cores per worker.
sat_tier = int(budget_ram / CORES / (kbpn / 1024.0 / 1024.0)) if kbpn > 0 else 0
print(f"RAM-BALANCED TIER for this box: ~{sat_tier:,} nodes/worker saturates all {CORES} cores at ~{budget_ram:.0f} GB.")
print(f"  Below it RAM sits idle ({CORES} cores @ 200k only touch ~{CORES*200000*kbpn/1024/1024:.0f} GB); above it you")
print(f"  go RAM-capped and idle cores. PROBE_WORKERS=None already auto-fills cores within the RAM budget.\n")

def usable_workers(tier):
    rss_gb = kbpn * tier / 1024.0 / 1024.0
    by_ram = int(budget_ram / rss_gb) if rss_gb > 0 else CORES
    return max(1, min(CORES, by_ram)), rss_gb, by_ram

if nps_est > 0:
    print("PROJECTED SWEEP TIME (per z=w arm; the fleet runs one arm per box in parallel):")
    hdr = f"  {'dataset':18} {'#unsolved':>9} {'tier':>11} {'GB/worker':>9} {'workers':>8} {'wall hrs/arm':>12}"
    print(hdr); print("  " + "-" * (len(hdr) - 2))
    for label, ntot, nuns in PROJECT_OVER:
        for tier in CANDIDATE_BUDGETS:
            w, rss_gb, by_ram = usable_workers(tier)
            wall_h = tier * nuns / nps_est / 3600.0 / w
            note = "" if by_ram >= CORES else f"  (RAM-capped: {by_ram} of {CORES} cores)"
            print(f"  {label:18} {nuns:9} {tier:11,} {rss_gb:9.1f} {w:8} {wall_h:12.1f}{note}")
    print()
    print(f"  Reading it: 'workers' = min(cores, RAM/GB-per-worker); 'wall hrs/arm' already divides by it.")
    print(f"  One arm per box in parallel -> FLEET wall ~= the largest 'wall hrs/arm'. Whole experiment")
    print(f"  COMPUTE = that x {N_ARMS_TOTAL} arms. If a tier is RAM-capped, that's the lever to watch")
    print(f"  (bigger box, or shard the idx range across boxes). Throughput is ~depth-stable, so a 200k")
    print(f"  probe projects the larger tiers reasonably; confirm memory by probing near the target tier.")


probes: 1029 hard, 11 easy   (arms: ['r1', 'r2', 'x', 'y'], runs: ['?', 'r1_b500k_w6_20260703-012249', 'r1_b500k_w6_20260703-042924', 'r1_b500k_w6_20260703-051042', 'r2_b500k_w6_20260703-012855', 'r2_b500k_w6_20260703-050717', 'r2_b500k_w6_20260703-054835', 'smoke', 'x_b500k_w6_20260703-012915', 'x_b500k_w6_20260703-050740', 'x_b500k_w6_20260703-054825', 'y_b500k_w6_20260703-013000', 'y_b500k_w6_20260703-050748', 'y_b500k_w6_20260703-054850'])

THROUGHPUT (nodes/sec, per core):
  hard all      : median 594  min 232  max 1251
  hard exhausting: median 594   <-- used for the projection (1029/1029 exhausted budget)
  easy solved   : median wall 0.06s (small addend)

MEMORY (per worker): peak at 500,000 nodes = 7687 MB ; rate ~10.9 KB/node
  box RAM 51 GB ; planning against 41 GB (10 GB safety margin left free)
RAM-BALANCED TIER for this box: ~492,071 nodes/worker saturates all 8 cores at ~41 GB.
  Below it RAM sits idle (8 cores @ 200k only touch ~17 GB); above it you
  go RAM-capped and 

In [8]:
# ==================== VERIFY PERSISTED SOLVE PATHS (independent replay + decoded moves, ALL runs) ====================
# Re-reads every results/paths_*.jsonl (local + Drive) and re-runs gn.verify_path on each solved path
# (recomputes neighbors from scratch -- does NOT trust the run-time boolean), then prints the exact
# substitution moves for one example.
_LET = {0: "1", 1: "x", 2: "y", 3: "z", -1: "X", -2: "Y", -3: "Z"}   # empty/pad word shown as trivial gen "1"
def _word(r):
    return "".join(_LET.get(int(a), "?") for a in r) or "1"

paths = {}                                  # dedup by (dataset,idx,arm) across all files; last wins
for r in read_jsonl(result_files("paths")):
    try: paths[(r.get("dataset"), r["idx"], r.get("arm"))] = r
    except Exception: pass
sol = list(paths.values())
print(f"{len(sol)} solved path(s) across {len(result_files('paths'))} paths file(s)\n")

allok = True
for r in sol:
    states = gn.deserialize_states(r)
    ok = gn.verify_path(states, r.get("n_gen", N_GEN))
    allok &= bool(ok)
    print(f"  {str(r.get('dataset')):16} idx {r['idx']:4} z={r.get('arm')} [{r.get('run_tag','?')}]: "
          f"path_len={r['path_len']:3}  final_trivial={gn.is_trivial(states[-1])}  verify_path={ok}")
print("\n" + ("ALL PATHS VERIFIED (independent replay -> trivial)" if (sol and allok)
              else ("SOME PATHS FAILED -- investigate" if sol else "no paths yet -- run the RUN cell first")))

# exact substitution moves for the first solved example, letter-decoded (x y z / X Y Z)
if sol:
    r = sol[0]; states, moves = r["states"], r["moves"]
    print(f"\n--- moves: {r.get('dataset')} idx {r['idx']} (z={r.get('arm')}), {r['path_len']} steps ---")
    print(f"  start : {[_word(w) for w in states[0]]}")
    for t in range(1, len(states)):
        m = moves[t]                        # (relator_slot_changed, rot_a, rot_c, c_is_inverse) or None
        print(f"  {t:3} {str(tuple(m)) if m else 'None':>18} : {[_word(w) for w in states[t]]}")
    print("  (move = (slot_changed, rot_a, rot_c, c_is_inverse); shown state is canonical after the move.)")


6 solved path(s) across 3 paths file(s)

  1190MS           idx    0 z=r2 [r2_b500k_w6_20260703-054835]: path_len=  3  final_trivial=True  verify_path=True
  1190MS           idx  320 z=r2 [r2_b500k_w6_20260703-054835]: path_len= 20  final_trivial=True  verify_path=True
  1190MS           idx    0 z=r1 [smoke]: path_len=  5  final_trivial=True  verify_path=True
  1190MS           idx  320 z=r1 [smoke]: path_len= 19  final_trivial=True  verify_path=True
  1190MS           idx    0 z=y [y_b500k_w6_20260703-054850]: path_len=  3  final_trivial=True  verify_path=True
  1190MS           idx  320 z=y [y_b500k_w6_20260703-054850]: path_len= 18  final_trivial=True  verify_path=True

ALL PATHS VERIFIED (independent replay -> trivial)

--- moves: 1190MS idx 0 (z=r2), 3 steps ---
  start : ['Yx', 'ZYx', 'YYXyx']
    1       (2, 0, 0, 1) : ['X', 'Yx', 'ZYx']
    2       (2, 0, 0, 1) : ['Z', 'X', 'Yx']
    3       (2, 0, 1, 0) : ['Z', 'Y', 'X']
  (move = (slot_changed, rot_a, rot_c, c_is_inverse); 